In [5]:
"""
PIPELINE STAGE: Feature Engineering & Spatio-Temporal Enrichment
INPUT: 15_final.xlsx (Pre-processed energy data)
OUTPUT: ML_En.xlsx (Master dataset for Machine Learning)
LIBRARIES: pandas, os

1. OBJECTIVE:
    Transform the consolidated energy dataset into a high-resolution master file by 
    injecting spatio-temporal features (Season & Region) and standardizing the 
    schema for PeerJ journal submission standards.

2. PRE-PROCESSING & NORMALIZATION:
    - Case-Insensitive Alignment: Convert all incoming headers to lowercase and 
      strip whitespaces to ensure 100% matching reliability for 'Daylight' and 'Plate' columns.

3. TEMPORAL ENCODING (Season Extraction):
    - Implementation of a seasonal mapping (1-4):
        * 1: Spring (March, April, May)
        * 2: Summer (June, July, August)
        * 3: Autumn (September, October, November)
        * 4: Winter (December, January, February)

4. SPATIAL ENCODING (Regional Categorization):
    - Strategic Mapping: Assign each 'Plate' (Province Code) to its respective 
      geographical region (1-7) based on the official Turkish regional division:
      (1: Marmara, 2: Ege, 3: Akdeniz, 4: Karadeniz, 5: İç Anadolu, 6: Doğu Anadolu, 7: Güneydoğu Anadolu).
    - Reverse Mapping Technique: Convert lists into a flat dictionary for optimized 
      O(1) look-up performance during 'Region' column assignment.

5. SCHEMA STANDARDIZATION & FEATURE ORDERING:
    - Final Renaming: Re-map technical labels into professional publication headers 
      (e.g., 'industrial' -> 'Industry', 'daylight' -> 'Daylight Duration').
    - Precision Ordering: Organize the final matrix in a logical flow:
      [Spatio-Temporal Keys] -> [Target Consumption] -> [Consumer Type Features] -> [Demographics & Geospatial Metrics].

6. RESILIENCE & LOGGING:
    - Implement a production-ready 'try-except' block for error handling.
    - Automated Directory Management: Use 'os.makedirs' to ensure the output path 
      exists before saving.
"""

import pandas as pd
import os

def final_data_enrichment():
    # 1. Dosya Yolları
    INPUT_PATH = os.path.join("..", "..", "processed_data", "steps", "15_final.xlsx")
    OUTPUT_PATH = os.path.join("..", "..", "processed_data", "final", "ML_En.xlsx")

    try:
        # 2. Dosyayı Yükle
        print("Veri seti yükleniyor...")
        df = pd.read_excel(INPUT_PATH)

        # 3. Sütun İsimlerini Normalize Et (Eşleşme hatalarını önlemek için)
        df.columns = df.columns.str.lower().str.strip()

        # 4. Mevsim (Season) Sütunu Ekleme
        season_map = {
            3: 1, 4: 1, 5: 1,    # İlkbahar
            6: 2, 7: 2, 8: 2,    # Yaz
            9: 3, 10: 3, 11: 3,  # Sonbahar
            12: 4, 1: 4, 2: 4    # Kış
        }
        print("Mevsimsel kodlama yapılıyor...")
        df['season'] = df['month'].map(season_map)

        # 5. Bölge (Region) Sütunu Ekleme (Sadece Sayısal No: 1-7)
        region_map_raw = {
            1: [10, 11, 16, 17, 22, 34, 39, 41, 54, 59, 77], # Marmara
            2: [3, 9, 20, 35, 43, 45, 48, 64],              # Ege
            3: [1, 7, 15, 31, 32, 33, 46, 80],              # Akdeniz
            4: [5, 8, 14, 19, 28, 29, 37, 52, 53, 55, 57, 60, 61, 67, 69, 74, 78, 81], # Karadeniz
            5: [6, 18, 26, 38, 40, 42, 50, 51, 58, 66, 68, 70, 71], # İç Anadolu
            6: [4, 12, 13, 23, 24, 25, 30, 36, 44, 49, 62, 65, 75, 76], # Doğu Anadolu
            7: [2, 21, 27, 47, 56, 63, 72, 73, 79]          # Güneydoğu Anadolu
        }
        
        # Plaka -> Bölge No eşleşmesini oluştur
        plate_to_region = {plaka: b_no for b_no, plakalar in region_map_raw.items() for plaka in plakalar}
        
        print("Bölgesel No ataması yapılıyor...")
        df['region'] = df['plate'].map(plate_to_region)

        # 6. Başlıkları Yeniden İsimlendirme
        final_rename = {
            'plate': 'Plate', 'year': 'Year', 'month': 'Month', 
            'season': 'Season', 'region': 'Region',
            'consumption': 'Consumption', 'lighting': 'Lighting',
            'public': 'Public', 'residential': 'Residential', 
            'industrial': 'Industry', 'agricultural': 'Agriculture',
            'population': 'Population', 'latitude': 'Latitude', 
            'longitude': 'Longitude', 'daylight': 'Daylight Duration', 
            'altitude': 'Altitude'
        }
        
        df = df.rename(columns=final_rename)

        # 7. İstenen Tam Sıra ve Filtreleme
        target_order = [
            'Plate', 'Year', 'Month', 'Season', 'Region', 
            'Consumption', 'Lighting', 'Public', 'Residential', 
            'Industry', 'Agriculture', 'Population', 
            'Latitude', 'Longitude', 'Daylight Duration', 'Altitude'
        ]
        
        # Sadece mevcut sütunları target_order sırasına göre seç
        df = df[[col for col in target_order if col in df.columns]]

        # 8. Kaydet
        os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
        df.to_excel(OUTPUT_PATH, index=False)
        
        print("\n" + "="*60)
        print("İŞLEM TAMAMLANDI: MASTER DATASET HAZIRLANDI")
        print(f"Çıktı Dosyası: {OUTPUT_PATH}")
        print("Eklenen/Düzenlenen: Season (1-4), Region (1-7), Daylight Duration")
        print("="*60)
        print(df.head())

    except Exception as e:
        print(f"HATA: {e}")

if __name__ == "__main__":
    final_data_enrichment()

Veri seti yükleniyor...
Mevsimsel kodlama yapılıyor...
Bölgesel No ataması yapılıyor...

İŞLEM TAMAMLANDI: MASTER DATASET HAZIRLANDI
Çıktı Dosyası: ..\..\processed_data\final\ML_En.xlsx
Eklenen/Düzenlenen: Season (1-4), Region (1-7), Daylight Duration
   Plate  Year  Month  Season  Region  Consumption  Lighting  Public  \
0      1  2020      1       4       3       610866      7938  166466   
1      1  2020      2       4       3       572566      8033  166949   
2      1  2020      3       1       3       538521      8065  167515   
3      1  2020      4       1       3       434992      8077  167675   
4      1  2020      5       1       3       466147      8094  168197   

   Residential  Industry  Agriculture  Population  Latitude  Longitude  \
0       889243      1881        16039     2258718      37.0  35.321333   
1       892564      1921        16098     2258718      37.0  35.321333   
2       896263      1940        16175     2258718      37.0  35.321333   
3       897588     